<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loss

This notebook contains PyTorch classes that can be used as loss functions for segmentation tasks

## Imports

In [ ]:
import torch
import torch.nn as nn

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/metrics.ipynb"

## Dice Loss

In [ ]:
class DiceLoss(nn.Module):
  def __init__(self, smooth=1.0):
    super().__init__()
    self.metric = DiceCoefficient(smooth=smooth)

  def forward(self, pred_y, ground_y):
    dice_coefficient = self.metric(pred_y, ground_y)

    return 1 - dice_coefficient

## IoU Loss

In [ ]:
class IoULoss(nn.Module):
  def __init__(self, smooth=1.0):
    super().__init__()
    self.metric = IoU(smooth=smooth)

  def forward(self, pred_y, ground_y):
    iou = self.metric(pred_y, ground_y)

    return 1 - iou

## Weighted Dice Loss

In [ ]:
class WeightedTrainDiceCoefficient(nn.Module):
  def __init__(self, smooth=1.0):
    super().__init__()
    self.smooth = smooth

  def forward(self, pred_y, ground_y):
    flat_pred_y = pred_y.reshape(-1)
    flat_ground_y = ground_y.reshape(-1)

    intersection = (flat_pred_y * flat_ground_y).sum()

    return (2 * intersection + self.smooth) / (flat_pred_y.sum() + flat_ground_y.sum() + self.smooth)

  def get_name() -> str:
    return "dice_coefficient"

In [ ]:
class WeightedTrainDiceLoss(nn.Module):
  def __init__(self, class_weights, smooth=1.0):
    super().__init__()
    self.metric = WeightedTrainDiceCoefficient(smooth=smooth)
    self.class_weights = class_weights

    assert sum(self.class_weights) == 1

  def forward(self, pred_y, ground_y):
    total_loss = 0

    for class_idx, class_weight in enumerate(self.class_weights):
      dice_coefficient = self.metric(pred_y[:, class_idx], ground_y[:, class_idx])
      loss = class_weight * (1 - dice_coefficient)
      total_loss += loss

    return total_loss